# Building a 1-D CNN (Temporal Convolution Network) Classifier

> Imports


In [14]:
%load_ext autoreload
%autoreload 2

from helpers import plotLightCurveFromDF, saveLightCurveFromDF, sampleRandomKIC
import lightkurve as lk
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import gc
from sklearn.preprocessing import LabelEncoder

%matplotlib inline


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
df = pd.read_parquet("./local-assets/MERGED_LCS.parquet")
df


,time,flux,Class
KIC,,,
757450,"[131.51271468758932, 131.5331494016791, 131.55...","[1.0392150925008297, 1.0383358659303283, 1.038...",CONFIRMED
892772,"[352.39657371611975, 352.43743948176416, 352.4...","[1.019566672226177, 1.0188446568700726, 1.0189...",FALSE POSITIVE
1025986,"[120.53931629149156, 120.55975094284076, 120.5...","[1.0488531548750866, 1.0523113529281902, 1.051...",CANDIDATE
1026032,"[131.5127135392686, 131.53314824846893, 131.55...","[1.0509398498643578, 1.0503349421979056, 1.050...",FALSE POSITIVE
1026957,"[120.53930025400041, 120.55973490802717, 120.5...","[1.0599377280481261, 1.0591694377327983, 1.058...",CONFIRMED
...,...,...,...
202140012,"[1940.0083829938958, 1940.0288146129678, 1940....","[0.9961692094802856, 0.9965322613716125, 0.996...",VARIABLE STAR
202140013,"[1940.0083988461702, 1940.0288304660571, 1940....","[0.9897362589836121, 0.9907680153846741, 0.991...",VARIABLE STAR
202140059,"[1940.0088655664877, 1940.0292972054667, 1940....","[1.0229097604751587, 1.014359951019287, 1.0042...",FALSE POSITIVE


In [3]:
df = df.reset_index()


---

> Rename features


In [4]:
renameMap = {
    'time': 'Time',
    'flux': 'Flux'
}

df = df.rename(columns=renameMap)
df


,KIC,Time,Flux,Class
0,757450,"[131.51271468758932, 131.5331494016791, 131.55...","[1.0392150925008297, 1.0383358659303283, 1.038...",CONFIRMED
1,892772,"[352.39657371611975, 352.43743948176416, 352.4...","[1.019566672226177, 1.0188446568700726, 1.0189...",FALSE POSITIVE
2,1025986,"[120.53931629149156, 120.55975094284076, 120.5...","[1.0488531548750866, 1.0523113529281902, 1.051...",CANDIDATE
3,1026032,"[131.5127135392686, 131.53314824846893, 131.55...","[1.0509398498643578, 1.0503349421979056, 1.050...",FALSE POSITIVE
4,1026957,"[120.53930025400041, 120.55973490802717, 120.5...","[1.0599377280481261, 1.0591694377327983, 1.058...",CONFIRMED
...,...,...,...,...
18877,202140012,"[1940.0083829938958, 1940.0288146129678, 1940....","[0.9961692094802856, 0.9965322613716125, 0.996...",VARIABLE STAR
18878,202140013,"[1940.0083988461702, 1940.0288304660571, 1940....","[0.9897362589836121, 0.9907680153846741, 0.991...",VARIABLE STAR
18879,202140059,"[1940.0088655664877, 1940.0292972054667, 1940....","[1.0229097604751587, 1.014359951019287, 1.0042...",FALSE POSITIVE
18880,202140094,"[1940.0088592208049, 1940.029290861763, 1940.0...","[1.008090615272522, 1.0085946321487427, 1.0099...",FALSE POSITIVE


---

> Remove duplicate KIC entries

In [5]:
def chooseLabel(labels):
    labels = list(labels)
    nonFp = [lab for lab in labels if lab != "FALSE POSITIVE"]

    if len(nonFp) > 0:
        # If there are multiple non-FP labels, take the most common one
        return pd.Series(nonFp).mode().iloc[0]

    # If all labels are FALSE POSITIVE, keep one of them
    return pd.Series(labels).mode().iloc[0]


In [6]:
df = (
    df.groupby("KIC", as_index=False)
        .agg({
            "Time": "first",
            "Flux": "first",
            "Class": chooseLabel
        })
)


In [7]:
df['Class'].value_counts()


Class
FALSE POSITIVE           7522
ECLIPSING BINARY STAR    3049
VARIABLE STAR            2980
CONFIRMED                1972
CANDIDATE                1637
Name: count, dtype: int64

In [8]:
df.duplicated(subset="KIC").sum()


np.int64(0)

---

> Remove 'CANDIDATE' class

In [11]:
df = df[df['Class'] != 'CANDIDATE'].copy()


In [13]:
df


,KIC,Time,Flux,Class
0,757450,"[131.51271468758932, 131.5331494016791, 131.55...","[1.0392150925008297, 1.0383358659303283, 1.038...",CONFIRMED
1,892772,"[352.39657371611975, 352.43743948176416, 352.4...","[1.019566672226177, 1.0188446568700726, 1.0189...",FALSE POSITIVE
3,1026032,"[131.5127135392686, 131.53314824846893, 131.55...","[1.0509398498643578, 1.0503349421979056, 1.050...",ECLIPSING BINARY STAR
4,1026957,"[120.53930025400041, 120.55973490802717, 120.5...","[1.0599377280481261, 1.0591694377327983, 1.058...",CONFIRMED
5,1027438,"[131.51268857505056, 131.53312328704487, 131.5...","[1.0649623994019086, 1.0653504391733866, 1.065...",FALSE POSITIVE
...,...,...,...,...
17155,202140012,"[1940.0083829938958, 1940.0288146129678, 1940....","[0.9961692094802856, 0.9965322613716125, 0.996...",VARIABLE STAR
17156,202140013,"[1940.0083988461702, 1940.0288304660571, 1940....","[0.9897362589836121, 0.9907680153846741, 0.991...",VARIABLE STAR
17157,202140059,"[1940.0088655664877, 1940.0292972054667, 1940....","[1.0229097604751587, 1.014359951019287, 1.0042...",FALSE POSITIVE
17158,202140094,"[1940.0088592208049, 1940.029290861763, 1940.0...","[1.008090615272522, 1.0085946321487427, 1.0099...",FALSE POSITIVE


---

> Encode 'Class' labels

In [15]:
def encodeLabels(df):
    """
    Function to encode the labels in the 'Class' feature of a given dataframe df.
    """
    le = LabelEncoder()
    y = le.fit_transform(df["Class"].values)
    df = df.copy()
    df["label"] = y

    return df, le


In [17]:
df, le = encodeLabels(df)
df


,KIC,Time,Flux,Class,label
0,757450,"[131.51271468758932, 131.5331494016791, 131.55...","[1.0392150925008297, 1.0383358659303283, 1.038...",CONFIRMED,0
1,892772,"[352.39657371611975, 352.43743948176416, 352.4...","[1.019566672226177, 1.0188446568700726, 1.0189...",FALSE POSITIVE,2
3,1026032,"[131.5127135392686, 131.53314824846893, 131.55...","[1.0509398498643578, 1.0503349421979056, 1.050...",ECLIPSING BINARY STAR,1
4,1026957,"[120.53930025400041, 120.55973490802717, 120.5...","[1.0599377280481261, 1.0591694377327983, 1.058...",CONFIRMED,0
5,1027438,"[131.51268857505056, 131.53312328704487, 131.5...","[1.0649623994019086, 1.0653504391733866, 1.065...",FALSE POSITIVE,2
...,...,...,...,...,...
17155,202140012,"[1940.0083829938958, 1940.0288146129678, 1940....","[0.9961692094802856, 0.9965322613716125, 0.996...",VARIABLE STAR,3
17156,202140013,"[1940.0083988461702, 1940.0288304660571, 1940....","[0.9897362589836121, 0.9907680153846741, 0.991...",VARIABLE STAR,3
17157,202140059,"[1940.0088655664877, 1940.0292972054667, 1940....","[1.0229097604751587, 1.014359951019287, 1.0042...",FALSE POSITIVE,2
17158,202140094,"[1940.0088592208049, 1940.029290861763, 1940.0...","[1.008090615272522, 1.0085946321487427, 1.0099...",FALSE POSITIVE,2
